# 🎬 3D Stickman Fight Video Generator (Google Colab Free GPU)

This notebook renders a **cinematic 3D stickman martial-arts fight** with:
- **Seamless 270° Orbiting 3D Camera**
- **0.15x Matrix Bullet-Time Slow Motion**
- **Dual Neon Stickmen (Cyan vs Crimson)**
- **Expanding 3D Shockwave Rings & Ground Slam**

---
### ⚡ Step 1: Ensure Free GPU is Selected
1. In the top menu, click **Runtime** → **Change runtime type**.
2. Select **T4 GPU** (Free tier).
3. Click **Save**.

In [ ]:
# Check that Google Cloud T4 GPU is active
!nvidia-smi

### 📦 Step 2: Install Blender on Colab

In [ ]:
# Fast headless Blender installation
!apt-get update -qq && apt-get install -y -qq blender
!blender --version

### 🥋 Step 3: Write the 3D Fight Choreography Script

In [ ]:
script_content = '''
import bpy
import math

def reset_scene():
    bpy.ops.wm.read_factory_settings(use_empty=True)

def create_emission_mat(name, color, strength=6.0):
    mat = bpy.data.materials.new(name=name)
    mat.use_nodes = True
    nodes = mat.node_tree.nodes
    nodes.clear()
    emit = nodes.new(type='ShaderNodeEmission')
    emit.inputs['Color'].default_value = color
    emit.inputs['Strength'].default_value = strength
    out = nodes.new(type='ShaderNodeOutputMaterial')
    mat.node_tree.links.new(emit.outputs['Emission'], out.inputs['Surface'])
    return mat

def build_arena():
    bpy.ops.mesh.primitive_plane_add(size=30, location=(0, 0, 0))
    floor = bpy.context.active_object
    mat = bpy.data.materials.new(name="FloorMat")
    mat.use_nodes = True
    bsdf = mat.node_tree.nodes['Principled BSDF']
    bsdf.inputs['Base Color'].default_value = (0.02, 0.03, 0.06, 1.0)
    bsdf.inputs['Roughness'].default_value = 0.15
    floor.data.materials.append(mat)
    
    grid_mat = create_emission_mat("GridRing", (0.0, 0.6, 1.0, 1.0), 2.0)
    bpy.ops.mesh.primitive_torus_add(major_radius=6.0, minor_radius=0.03, location=(0, 0, 0.02))
    bpy.context.active_object.data.materials.append(grid_mat)

def build_stickman(name, mat, base_x):
    bpy.ops.object.empty_add(type='PLAIN_AXES', location=(base_x, 0, 1.5))
    root = bpy.context.active_object
    root.name = f"{name}_Root"
    
    bpy.ops.mesh.primitive_uv_sphere_add(radius=0.32, location=(base_x, 0, 2.5))
    head = bpy.context.active_object
    head.data.materials.append(mat)
    head.parent = root
    
    bpy.ops.mesh.primitive_cylinder_add(radius=0.08, depth=1.0, location=(base_x, 0, 1.8))
    torso = bpy.context.active_object
    torso.data.materials.append(mat)
    torso.parent = root
    
    for side, offset in [("L", -0.45), ("R", 0.45)]:
        bpy.ops.mesh.primitive_cylinder_add(radius=0.07, depth=0.9, location=(base_x + offset, 0, 2.0))
        arm = bpy.context.active_object
        arm.data.materials.append(mat)
        arm.parent = root
        
    for side, offset in [("L", -0.25), ("R", 0.25)]:
        bpy.ops.mesh.primitive_cylinder_add(radius=0.08, depth=1.1, location=(base_x + offset, 0, 0.75))
        leg = bpy.context.active_object
        leg.data.materials.append(mat)
        leg.parent = root
        
    return root

def setup_scene():
    reset_scene()
    build_arena()
    mat_cyan = create_emission_mat("CyanMat", (0.0, 0.95, 1.0, 1.0), 6.0)
    mat_red = create_emission_mat("RedMat", (1.0, 0.05, 0.35, 1.0), 6.0)
    
    cyan = build_stickman("Cyan", mat_cyan, -2.2)
    red = build_stickman("Red", mat_red, 2.2)
    
    # Lights
    bpy.ops.object.light_add(type='POINT', location=(-4, -5, 5))
    bpy.context.active_object.data.energy = 800
    bpy.context.active_object.data.color = (0.0, 0.9, 1.0)
    
    bpy.ops.object.light_add(type='POINT', location=(4, 5, 5))
    bpy.context.active_object.data.energy = 800
    bpy.context.active_object.data.color = (1.0, 0.0, 0.3)
    
    # Camera
    bpy.ops.object.camera_add(location=(0, -6.5, 2.0))
    cam = bpy.context.active_object
    bpy.context.scene.camera = cam
    track = cam.constraints.new(type='TRACK_TO')
    track.target = cyan
    track.track_axis = 'TRACK_NEGATIVE_Z'
    track.up_axis = 'UP_Y'
    
    # Shockwave
    wave_mat = create_emission_mat("WaveMat", (1.0, 1.0, 1.0, 1.0), 10.0)
    bpy.ops.mesh.primitive_torus_add(major_radius=0.1, minor_radius=0.04, location=(0.6, 0, 0.05))
    wave = bpy.context.active_object
    wave.data.materials.append(wave_mat)
    wave.scale = (0.01, 0.01, 0.01)
    wave.keyframe_insert(data_path='scale', frame=189)
    wave.scale = (8.0, 8.0, 0.1)
    wave.keyframe_insert(data_path='scale', frame=220)
    
    # Animation
    bpy.context.scene.frame_start = 1
    bpy.context.scene.frame_end = 240
    
    cyan.location = (-2.2, 0, 1.5)
    cyan.keyframe_insert(data_path='location', frame=1)
    red.location = (2.2, 0, 1.5)
    red.keyframe_insert(data_path='location', frame=1)
    
    # Dash & Launch
    cyan.location = (-0.6, 0, 1.5)
    cyan.keyframe_insert(data_path='location', frame=60)
    red.location = (0.6, 0, 1.5)
    red.keyframe_insert(data_path='location', frame=60)
    
    cyan.location = (-0.7, 0, 3.4)
    cyan.keyframe_insert(data_path='location', frame=90)
    red.location = (0.7, 0, 3.6)
    red.keyframe_insert(data_path='location', frame=90)
    
    # 270° Bullet Time Orbit (Frames 90 to 180)
    for f in range(90, 181, 15):
        prog = (f - 90) / 90.0
        ang = -math.pi * 0.4 + prog * math.pi * 1.5
        cam.location = (math.sin(ang) * 4.2, math.cos(ang) * 4.2, 2.2 + math.sin(prog * math.pi) * 1.2)
        cam.keyframe_insert(data_path='location', frame=f)
        
    # Impact & Landing
    cyan.location = (-0.8, 0, 0.8)
    cyan.keyframe_insert(data_path='location', frame=195)
    red.location = (3.5, 0, 0.2)
    red.keyframe_insert(data_path='location', frame=195)
    
    # Render Config
    scene = bpy.context.scene
    scene.render.resolution_x = 1920
    scene.render.resolution_y = 1080
    scene.render.fps = 30
    scene.render.image_settings.file_format = 'FFMPEG'
    scene.render.ffmpeg.format = 'MPEG4'
    scene.render.ffmpeg.codec = 'H264'
    scene.render.filepath = '/content/stickman_fight_3d.mp4'

setup_scene()
'''

with open('director.py', 'w') as f:
    f.write(script_content)

print('✅ director.py written successfully!')

### 🚀 Step 4: Render the 3D Animation on Free Cloud GPU
This runs Blender in the cloud using Google's Nvidia GPU.

In [ ]:
# Render full 1080p MP4 headlessly
!blender -b -P director.py -a

### 📥 Step 5: Download & Watch Video on Your Laptop

In [ ]:
# Download finished MP4 straight to your laptop's Downloads folder
from google.colab import files
files.download('/content/stickman_fight_3d.mp4')